In [1]:
import os
import json


task_run = '''
19cd4d804ea6:d7fd3743be7d
'''

task = task_run.split(":")[0].replace('\n','')
run = task_run.split(":")[1].replace('\n','')

artifact_dir = f"data/tasks/{task}/runs/{run}"

module_map = dict()

schematic_file_names = os.listdir(f"{artifact_dir}/kicad/modules")
for file_name in schematic_file_names:
    if file_name.endswith(".kicad_sch"):
        module_id = file_name.split(".")[0]
        module_map[module_id] = f"{artifact_dir}/kicad/modules/{file_name}"

# output_path = f"{artifact_dir}/kicad/full_circuit.kicad_sch"

output_path = f"exported/test_{task}/full_circuit.kicad_sch"

module_map

blueprint_path = f"{artifact_dir}/blueprint.json"
with open(blueprint_path, "r") as f:
    bp = json.load(f)


In [2]:
from python.top_schematic_from_contract import generate_top_schematic_from_contract
result = generate_top_schematic_from_contract(
    module_sheets=module_map,
    rails=bp["rails"],
    signals=bp["signals"],
    output_path=output_path,
    export_svg=True
)

result

{'success': True,
 'schematic_path': 'exported/test_19cd4d804ea6/full_circuit.kicad_sch',
 'modules_count': 6,
 'svg_dir': 'exported/test_19cd4d804ea6/full_circuit_svg',
 'svg_paths': ['exported/test_19cd4d804ea6/full_circuit_svg/full_circuit-ANALOG_INPUT_CONN.svg',
  'exported/test_19cd4d804ea6/full_circuit_svg/full_circuit-AUDIO_CORE.svg',
  'exported/test_19cd4d804ea6/full_circuit_svg/full_circuit-PWR_IN.svg',
  'exported/test_19cd4d804ea6/full_circuit_svg/full_circuit-SPEAKER_OUT.svg',
  'exported/test_19cd4d804ea6/full_circuit_svg/full_circuit-SUB_OUT_CONN.svg',
  'exported/test_19cd4d804ea6/full_circuit_svg/full_circuit-WIRELESS_SUB_CONN.svg',
  'exported/test_19cd4d804ea6/full_circuit_svg/full_circuit.svg'],
 'svg_path': 'exported/test_19cd4d804ea6/full_circuit_svg/full_circuit.svg'}

In [ ]:
from python.netlist_schematic_pipeline import generate_schematic_from_skidl_module

task_run = '''
05d294ff5c6a:86b588f8e42d
'''

module_id = "PWR_SEQ"

task = task_run.split(":")[0].replace('\n','')
run = task_run.split(":")[1].replace('\n','')

skidl_module_path=f"data/tasks/{task}/runs/{run}/skidl/modules/{module_id.lower()}.py"

subcircuit_name = skidl_module_path.split("/")[-1].split(".")[0].upper()
output_path = f"exported/test_{task}/{subcircuit_name.lower()}.kicad_sch"


res = generate_schematic_from_skidl_module(
    skidl_module_path=skidl_module_path,
    subcircuit_name=subcircuit_name,
    output_path=output_path,
    export_svg=True,
    verify=True,
    auto_cut_problem_nets=True,
    auto_cut_strategy="edge",
    auto_cut_iterations=4,
    auto_cut_max_nets=8,
    auto_cut_min_crossings=1,
    auto_cut_min_max_length_mm=None,
    auto_cut_min_max_backtrack_mm=None,
)
res


In [ ]:
import os
import json

report_path = "exported/schematic_export_report.json"

with open(report_path, "r") as f:
    report = json.load(f)

cases = []
for entry in report["results"]:
    if entry["status"] == "Generate failed":
        cases.append(entry)

for (i, case) in enumerate(cases):
    print(f"{i}: {case['task_id']} - {case['run_id']} - {case['subcircuit_name']}")
    print(f"  {case['skidl_module_path']}")
    

In [ ]:
from python.netlist_schematic_pipeline import generate_schematic_from_skidl_module

problems_to_solve = [0,5,7]
idx = 5

case_to_run = cases[idx]
task = case_to_run["task_id"]
run = case_to_run["run_id"]
subcircuit_name = case_to_run["subcircuit_name"]
skidl_module_path=f"data/tasks/{task}/runs/{run}/skidl/modules/{subcircuit_name.lower()}.py"
output_path = f"exported/test_{task}/{subcircuit_name.lower()}.kicad_sch"


res = generate_schematic_from_skidl_module(
    skidl_module_path=skidl_module_path,
    subcircuit_name=subcircuit_name,
    output_path=output_path,
    export_svg=True,
    verify=True,
    auto_cut_problem_nets=True,
    auto_cut_strategy="edge",
    auto_cut_iterations=2,
    auto_cut_max_nets=8,
    auto_cut_min_crossings=1,
    auto_cut_min_max_length_mm=None,
    auto_cut_min_max_backtrack_mm=None,
)
res


summary:


05d294ff5c6a - 86b588f8e42d - LDO_DDR_VDDQL
  data/tasks/05d294ff5c6a/runs/86b588f8e42d/skidl/modules/ldo_ddr_vddql.py

b525a415904a - 2d1a21fc657a - BATTERY
  data/tasks/b525a415904a/runs/2d1a21fc657a/skidl/modules/battery.py 

e873b1b62c93 - 346661c16ffa - USB_PORT
  data/tasks/e873b1b62c93/runs/346661c16ffa/skidl/modules/usb_port.py